<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.24em;text-transform:uppercase;color:#A20025;margin:0 0 12px;">NeurIPS 2026 Education Track&nbsp;&middot;&nbsp;Generate, then verify</div>

<h1 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:33px;line-height:1.12;margin:0 0 16px;color:#1A1416;letter-spacing:-.01em;">X0. The environment self-check</h1>

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:0 0 4px;">

<em style="color:#6B6461;font-size:14.5px;">Before any exercise. Run this once, top to bottom, and read the table it prints.</em>

When a notebook fails in the first five minutes, a learner cannot tell a broken
environment from a wrong answer. Half an hour disappears into package versions
before the lesson starts.

So this notebook runs first, and it answers exactly one question: can this machine
produce the numbers the rest of the set will ask you for? It takes under thirty
seconds, installs nothing, downloads nothing, and touches no GPU. Ten checks are
required and four are optional. A required check that fails prints the one action
that fixes it. An optional check that fails prints what it would have added and then
gets out of your way, because none of the thirteen tasks needs any of them.

The last required check is the interesting one. It recomputes a single number, the
pairwise diversity of twenty generated ligands under the RDKit topological
fingerprint, and compares it against the value committed with this bundle to the
sixteenth decimal place. That number is a sharp test of your RDKit version: it moves
if your version differs from the one the bundle was built with. The shipped ligands
carry no explicit hydrogens, so stripping them changes nothing here; hydrogen handling
moves the number only when hydrogens are present, which X6 draws out. Exercise X6 asks
you to compute it yourself. By the time you get there, this check has already proved
your machine can.

<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Objectives</strong>
<ol style="margin:8px 0 0;padding-left:22px;font-size:15.5px;color:#241c1d;">
<li style="margin:6px 0;">Confirm the bundle is intact and every cached input matches its committed checksum</li>
<li style="margin:6px 0;">Confirm the base environment can produce one canary number bit for bit</li>
<li style="margin:6px 0;">Learn which optional packages are absent and what they would have added</li>
</ol>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #6B6461;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#6B6461;">Runtime</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Under thirty seconds on a laptop CPU. No install, no download, no GPU, no API key.</span>
</div>

In [1]:
# Path anchor and toolkit import. This is the only setup cell in the notebook.
#
# The bundle looks like this:
#     NEURIPS2026_EDU_GENERATE_VERIFY/
#         gv_toolkit.py          <- the shared toolkit
#         data/                 <- every cached file the exercises read
#         exercises/            <- you are here
#         solutions/
# so the root is one level up from this notebook, and data/ hangs off the root.
# Resolving it positionally means the same cell works locally, on Colab, and in a
# grader's batch run, with no absolute path anywhere.
import sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name in {"exercises", "solutions"} else HERE
sys.path.insert(0, str(ROOT))

import gv_toolkit as gv

DATA = gv.DATA
print("bundle root:", ROOT.name, "| cached inputs:", len(gv.REQUIRED_FILES) + len(gv.SUPPORTING_FILES))

bundle root: NEURIPS2026_EDU_GENERATE_VERIFY | cached inputs: 25


<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;1</span>The ten required checks</h2>

Each check prints a single line: a status, the name of the thing tested, and either
the measured value or the one action that fixes it. Nothing here raises, so you get
the whole table even when something is wrong, which is the point. A traceback tells
you where Python stopped; a table tells you what your machine can and cannot do.

In [2]:
# The ten required checks. Each returns (ok, detail, remedy).
import hashlib, io, json, platform, sys

RESULTS = []


def record(name, ok, detail, remedy=""):
    RESULTS.append({"name": name, "ok": bool(ok), "detail": detail, "remedy": remedy})


# 1. Python version and the path anchor.
py_ok = sys.version_info >= (3, 10)
record("python >= 3.10", py_ok, platform.python_version(),
       "Create a fresh environment: conda create -n generate-verify python=3.11")
record("data path resolves", DATA.is_dir(), str(DATA.relative_to(ROOT.parent)),
       "Open this notebook from inside exercises/ in the unzipped bundle.")

# 2. Every cached input is present and matches its committed SHA-256.
manifest = json.loads((DATA / "MANIFEST.sha256.json").read_text())
expected = gv.REQUIRED_FILES + gv.SUPPORTING_FILES
missing = [f for f in expected if not (DATA / f).exists()]
bad = []
for f in expected:
    p = DATA / f
    if p.exists() and gv.sha256_file(p) != manifest[f]["sha256"]:
        bad.append(f)
record(f"all {len(expected)} cached inputs present", not missing,
       f"{len(expected) - len(missing)} of {len(expected)} found",
       "Re-download the bundle; missing: " + ", ".join(missing[:4]))
record("every checksum matches the manifest", not bad,
       f"{len(expected) - len(bad)} of {len(expected)} verified",
       "Re-download; altered: " + ", ".join(bad[:4]))

In [3]:
# 3. The provenance-safe reader returns the right shape, and the trap fires.
import pandas as pd

bz = gv.read_metrics_csv(DATA / "boltz2_comparison.csv")
pb = gv.read_metrics_csv(DATA / "posebusters_diffsbdd.csv")
shape_ok = (len(bz) == 7) and (len(pb) == 20)
record("provenance-safe CSV reader", shape_ok,
       f"boltz2_comparison {len(bz)} rows, posebusters_diffsbdd {len(pb)} rows",
       "Use gv.read_metrics_csv, which strips only the leading '#' lines.")

# The negative control. Passing comment='#' raises no exception and leaves NaN
# cells across the table. We trigger it once here so you see the damage before
# starting an exercise.
wrong = pd.read_csv(DATA / "boltz2_comparison.csv", comment="#")
n_nan = int(wrong.isna().sum().sum())
record("the comment='#' trap fires as expected", n_nan > 0,
       f"pandas comment='#' produces {n_nan} NaN cells in boltz2_comparison.csv "
       f"against {int(bz.isna().sum().sum())} with the correct reader",
       "This check passing is the healthy outcome. It confirms the hazard is real.")

In [4]:
# 4. RDKit round-trips a nitrile, which is the character behind the CSV trap.
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

NITRILE = "COC1C(=C=O)C(C#N)=NC2=C1C(c1ccc(Cl)cc1)N1CCCC1C2"   # DiffSBDD design 16
m = Chem.MolFromSmiles(NITRILE)
roundtrip = Chem.MolToSmiles(m) if m is not None else None
record("RDKit imports and round-trips a nitrile", roundtrip is not None,
       f"rdkit {Chem.rdBase.rdkitVersion}, {m.GetNumAtoms() if m else 0} heavy atoms",
       "conda install -c conda-forge rdkit")

# 5. torch imports and a two-layer MLP does one CPU forward pass.
import torch

torch.set_num_threads(4)
mlp = torch.nn.Sequential(torch.nn.Linear(2, 8), torch.nn.SiLU(), torch.nn.Linear(8, 2))
with torch.no_grad():
    y = mlp(torch.zeros(4, 2))
record("torch forward pass on CPU", tuple(y.shape) == (4, 2),
       f"torch {torch.__version__}, output shape {tuple(y.shape)}",
       "conda install -c conda-forge pytorch-cpu")

# 6. Both pinned toy checkpoints load and expose the expected state-dict keys.
ckpt_ok, ckpt_detail = True, []
for fname in ("toy_ddpm_trained.pt", "toy_fm_trained.pt"):
    blob = torch.load(DATA / fname, map_location="cpu", weights_only=False)
    keys = set(blob["model_state"])
    have = {"temb.dim"} <= keys or any(k.startswith("net.") for k in keys)
    ckpt_ok &= have and any(k.startswith("tproj.") for k in keys)
    ckpt_detail.append(f"{fname}: {len(keys)} tensors")
record("both toy checkpoints load", ckpt_ok, "; ".join(ckpt_detail),
       "Re-download data/toy_ddpm_trained.pt and data/toy_fm_trained.pt.")

In [5]:
# 7. The canary. One number, recomputed here, compared bit for bit.
#
# Pairwise diversity of the twenty pocket-conditioned ligands under the RDKit
# topological fingerprint at library defaults, over the 190 unordered pairs, with
# hydrogen handling pinned by Chem.RemoveHs. Every clause of that sentence changes
# the number, which is exactly what exercise X6 is about; do not go looking up the
# alternatives here, because working them out is the exercise.
CANARY = 0.8089776623453144

mols = gv.read_sdf(DATA / "diffsbdd_generated.sdf")
div, n_pairs = gv.pairwise_diversity(mols, fingerprint="rdkit")
exact = (div == CANARY)
record("canary reproduces bit for bit", exact,
       f"{div!r} over {n_pairs} pairs from {len(mols)} molecules",
       "If this is close but not exact, your RDKit version differs from the one "
       "the bundle was built with; the exercises still work, tolerances absorb it.")

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;2</span>The four optional checks</h2>

None of these blocks anything. Every one of the thirteen tasks runs without them.
Listing them up front shows which module walkthroughs gain a fresh-compute option
when those packages are installed.

In [6]:
# Optional. Absent is fine; the line says what it would have added.
import importlib

OPTIONAL = [
    ("posebusters", "recomputes the PoseBusters battery in module 4 instead of "
                    "reading the shipped table; X5 and X9 read the table either way"),
    ("posecheck", "recomputes strain and clash counts in module 4; X7 reads the "
                  "shipped table either way"),
    ("flowmol", "regenerates 3D molecules in module 3; X4 reads the shipped "
                "step-count table either way"),
]
for name, adds in OPTIONAL:
    try:
        importlib.import_module(name)
        print(f"  [ok]       {name:14s} available")
    except ImportError:
        print(f"  [absent]   {name:14s} {adds}")

gpu = torch.cuda.is_available()
print(f"  [{'ok' if gpu else 'absent'}]{'':<7}{'GPU':14s} "
      + ("visible; nothing in this set needs it"
         if gpu else "not visible; nothing in this set needs it"))

  [absent]   posebusters    recomputes the PoseBusters battery in module 4 instead of reading the shipped table; X5 and X9 read the table either way
  [absent]   posecheck      recomputes strain and clash counts in module 4; X7 reads the shipped table either way
  [absent]   flowmol        regenerates 3D molecules in module 3; X4 reads the shipped step-count table either way
  [absent]       GPU            not visible; nothing in this set needs it


<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;3</span>The verdict</h2>

In [7]:
# The table, then the one line that matters.
width = max(len(r["name"]) for r in RESULTS) + 2
print(f"{'check':<{width}} status   detail")
print("-" * (width + 60))
for r in RESULTS:
    status = "PASS" if r["ok"] else "FAIL"
    print(f"{r['name']:<{width}} {status:<8} {r['detail']}")
    if not r["ok"] and r["remedy"]:
        print(f"{'':<{width}} {'':<8} fix: {r['remedy']}")

n_ok = sum(r["ok"] for r in RESULTS)
print()
if n_ok == len(RESULTS):
    print("Required checks passed. You can run every module and every exercise "
          "on the cached CPU path.")
else:
    print(f"{len(RESULTS) - n_ok} required check(s) failed. Apply the fix lines "
          "above, then re-run this notebook before starting X1.")

check                                     status   detail
-----------------------------------------------------------------------------------------------------
python >= 3.10                            PASS     3.11.15
data path resolves                        PASS     NEURIPS2026_EDU_GENERATE_VERIFY/data
all 25 cached inputs present              PASS     25 of 25 found
every checksum matches the manifest       PASS     25 of 25 verified
provenance-safe CSV reader                PASS     boltz2_comparison 7 rows, posebusters_diffsbdd 20 rows
the comment='#' trap fires as expected    PASS     pandas comment='#' produces 16 NaN cells in boltz2_comparison.csv against 0 with the correct reader
RDKit imports and round-trips a nitrile   PASS     rdkit 2026.03.5, 26 heavy atoms
torch forward pass on CPU                 PASS     torch 2.13.0, output shape (4, 2)
both toy checkpoints load                 PASS     toy_ddpm_trained.pt: 12 tensors; toy_fm_trained.pt: 12 tensors
canary reproduces b

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #A20025;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Note</strong><br>
<span style="color:#3d3436;font-size:14.5px;">X0 has no answer to submit and no self-check key of its own. Its output is the table above. Once every required check reads PASS, open <code>EX1_sampling_budget_and_controls.ipynb</code>.</span>
</div>